# Ungraded Lab 2: LLM Calls and Crafting Simple Augmented Prompts
未评分实验 2：LLM 调用和制作简单的增强提示词


欢迎来到 LLM 调用和制作简单的增强提示词 实验。在本实验中，您将获得使用两个核心功能进行实践操作的机会，这些功能使您能够与大型语言模型 (LLM) 进行交互。这些功能既可以帮助您向 LLM 发送单个提示词 (single prompts)，也可以进行来回的对话 (back-and-forth conversation)。主要目标是向您展示如何向提示词中添加额外的信息，使其更详细、更有用。这种额外的上下文 (added context) 有助于模型为您提供更好、更精确的回复。

您将在本实验中学到：
如何设置并向 LLM 发送问题，包括处理单个问题和对话场景。

如何利用附加数据 (additional data) 来丰富您的提示词（即增强提示词），从而改进模型的回复。




# Table of Contents
- [ 1 - Understanding the functions to call LLMs](#1)
  - [ 1.1 `generate_with_single_input`](#1-1)
  - [ 1.2 `generate_with_multiple_input`](#1-2)
- [ 2 - Integrating Data into an LLM Prompt](#2)
  - [ 2.1 Understanding the data structure](#2-1)
  - [ 2.2 Creating the Prompt](#2-2)


In [3]:
!pip install together

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [together]5/6 [together]


In [6]:
from utils import (
    generate_with_single_input, 
    generate_with_multiple_input
)

<a id='1'></a>
## 1 - Understanding the functions to call LLMs

In this section you will explore the one function that will be used to call LLMs in this course. This function calls the [together.ai](https://www.together.ai/) API to call the models. Here in the Coursera environment some steps in reaching the Together API are handled on your behalf via a proxy server, so if you try to run this notebook outside the Coursera environment it won't work right away. With small adjustments, however, you can pass an optional parameter with a together.ai API key, which will allow you to run these notebooks on your local machine.

在本节中，您将探索本课程中用于调用大型语言模型（LLM）的一个函数。此函数通过调用 together.ai API 来使用模型。在 Coursera 环境中，访问 Together API 的某些步骤是通过代理服务器为您代为处理的。因此，如果您尝试在 Coursera 环境之外运行此 Notebook，它将无法立即工作。但是，只需进行少量调整，您就可以传入一个带有 together.ai API 密钥的可选参数，这将允许您在本地机器上运行这些 Notebook。



<a id='1-1'></a>
### 1.1 `generate_with_single_input`

This function allows you to generate text from a language model based on a single input prompt. For now, let's just focus on a simple call with only a few basic parameters. You will explore different parameters to call an LLM and their impact on the output in Module 4. Here's the parameters you'll have access to for now.

这个函数允许您根据单个输入提示词 (single input prompt) 从语言模型中生成文本。目前，我们只需关注一个只有少数基本参数的简单调用。您将在模块 4 中探索调用 LLM 的不同参数及其对输出的影响。现在，您将可以使用以下这些参数：

#### Parameters:

- `prompt` (str): The input text prompt you want to send to the language model.
- `max_tokens` (int): Maximum tokens to generate in the response.
- `model` (str): The model name. Default is `"meta-llama/Llama-3.2-3B-Instruct-Turbo"`.
- `together_api_key`: An optional API key for authentication; defaults to `None`. If `None` you will use our proxy, otherwise a direct call to together.ai will be performed with the provided API key.

提示词str(字符串)：您想要发送给语言模型的输入文本。

最大词元数int (整数)：模型在回复中将生成的最大词元（Token）数量。这用于限制输出的长度。

模型str(字符串)：要使用的模型名称。默认为 "meta-llama/Llama-3.2-3B-Instruct-Turbo"。

together_api_key：可选参数；用于身份验证的 API 密钥；默认为 None。如果为 None，您将使用课程提供的代理服务器；如果提供密钥，将直接调用 together.ai API。

In [7]:
# Example call
output = generate_with_single_input(
    prompt="What is the capital of France?"
)

print("Role:", output['role'])
print("Content:", output['content'])

/Users/a1-6/miniconda3/envs/ailearn/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.dlai.link'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Exception: Error while calling LLM: f<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again later, or contact the app or website owner.
<BR clear="all">
If you provide content to customers through CloudFront, you can find steps to troubleshoot and help prevent this error by reviewing the CloudFront documentation.
<BR clear="all">
<HR noshade size="1px">
<PRE>
Generated by cloudfront (CloudFront)
Request ID: vSpmCSNo2f17ADbIfTv731bPHZQmiz9mPTqm2ivWBKtEuNZUfh7kEw==
</PRE>
<ADDRESS>
</ADDRESS>
</BODY></HTML>

<a id='1-2'></a>
### 1.2 `generate_with_multiple_input`

This function is designed to handle multiple input messages in a conversational context. The input format is a dictionary with two keys:
这个函数旨在处理对话上下文中的多个输入消息。输入格式是一个包含两个键的字典：

1. 'role' - the role that the message is being passed (usually assistant, system or user)
2. 'content' - the prompt

'role'：传递消息的角色（通常是 assistant（助手/模型）、system（系统）或 user（用户））。
'content'：消息的提示词或内容。

#### Parameters:

- `messages` (List[Dict]): A list of dictionaries, each containing 'role' and 'content' keys to represent each message in the conversation.
- `max_tokens` (int): Determines token limit for the response.
- `model` (str): Model to be used, default is `"meta-llama/Llama-3.2-3B-Instruct-Turbo"`.

消息列表,List[Dict] (字典列表),一个字典列表，每个字典包含 'role' 和 'content' 两个键，用于表示对话中的每条消息。

最大词元数,int (整数),确定回复的词元限制。

模型,str (字符串),"要使用的模型名称，默认为 ""meta-llama/Llama-3.2-3B-Instruct-Turbo""。"

In [ ]:
# Example call
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

output = generate_with_multiple_input(
    messages=messages,
    max_tokens=100
)

print("Role:", output['role'])
print("Content:", output['content'])

### 1.3 Integration with OpenAI library

[Together.ai](together.ai) endpoints are [OpenAI compatible](https://docs.together.ai/docs/openai-api-compatibility) so you can use the [OpenAI library](https://github.com/openai/openai-python) to make the calls. In this section you will explore how to do it.  
Together.ai 的端点 (endpoints) 与 OpenAI 兼容，因此你可以使用 OpenAI 库 (library) 来进行调用。在本节中，你将探索如何实现这一点。


In [8]:
from openai import OpenAI, DefaultHttpxClient
import httpx

In [9]:
base_url = "http://proxy.dlai.link/coursera_proxy/together/" # If using together endpoint, add it here https://api.together.xyz/


# Custom transport to bypass SSL verification. This is only needed if using our proxy. Otherwise you can ignore it.
transport = httpx.HTTPTransport(local_address="0.0.0.0", verify=False)

# Create a DefaultHttpxClient instance with the custom transport
http_client = DefaultHttpxClient(transport=transport)

client = OpenAI(
    api_key = '', # Set any as our proxy does not use it. Set the together api key if using the together endpoint.
    base_url=base_url, 
   http_client=http_client, # ssl bypass to make it work via proxy calls, remove it if running with together.ai endpoint 
)


To use it, let's consider the same example as before.

In [10]:
messages = [
    {'role': 'user', 'content': 'Hello, who won the FIFA world cup in 2018?'},
    {'role': 'assistant', 'content': 'France won the 2018 FIFA World Cup.'},
    {'role': 'user', 'content': 'Who was the captain?'}
]

In [11]:
response = client.chat.completions.create(messages = messages, model ="meta-llama/Llama-3.2-3B-Instruct-Turbo")

PermissionDeniedError: <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">
<HTML><HEAD><META HTTP-EQUIV="Content-Type" CONTENT="text/html; charset=iso-8859-1">
<TITLE>ERROR: The request could not be satisfied</TITLE>
</HEAD><BODY>
<H1>403 ERROR</H1>
<H2>The request could not be satisfied.</H2>
<HR noshade size="1px">
Request blocked.
We can't connect to the server for this app or website at this time. There might be too much traffic or a configuration error. Try again later, or contact the app or website owner.
<BR clear="all">
If you provide content to customers through CloudFront, you can find steps to troubleshoot and help prevent this error by reviewing the CloudFront documentation.
<BR clear="all">
<HR noshade size="1px">
<PRE>
Generated by cloudfront (CloudFront)
Request ID: RPxicXiIydvDrrCSSWzuL7zTxw300zYO2s2mCHD7lK7CabCpi_KaZQ==
</PRE>
<ADDRESS>
</ADDRESS>
</BODY></HTML>

In [ ]:
print(response)

Notice that the response has several attributes. To access the response content, you may run:response.choices[0].message.content

In [ ]:
print(response.choices[0].message.content)

### 1.4 Note on Together.ai Integration for This Course

[Together.ai](https://together.ai) has generously provided credits for using the LLMs they host throughout this course. While there is technically a credit limit, it is set to be about 10 times more than what you would typically use, even with extensive usage. This is to ensure you have plenty of room to experiment and make your learning experience as smooth as possible.

There are two main types of errors you might encounter when making LLM calls during this course:

1. **500 and 429 Error**: This happens when too many calls are made to the system and it's overloaded. It's usually resolved by waiting for a moment. 
   
2. If you ever run out of credits, you will be notified. In this case, please reach out to our team in our Discourse community.

Grading your assignment will never use any of your credits. Our hope is that you never think need to think about the credit limit, and that in the unlikely situation that you hit it, you'll know what happened and we can resolve it for you rapidly.


Together.ai 非常慷慨地为本课程中使用的 LLM 提供了积分。虽然技术上存在积分限制，但其额度设置约为您通常使用量的 10 倍以上，即使您进行广泛的实验和使用，这也是为了确保您有足够的空间进行实验，并让您的学习体验尽可能顺利。

在本课程中进行 LLM 调用时，您可能会遇到两种主要的错误类型：

500 和 429 错误 (Error)：

原因：当系统调用量过大，导致系统超载 (overloaded) 时，就会发生此类错误。

解决：通常只需等待片刻即可自行解决。

积分用尽通知 (Out of credits)：

原因：如果您用完了所有积分，系统会通知您。

解决：在这种情况下，请通过我们的 Discourse 社区联系我们的团队。

对您作业的评分 (Grading) 绝不会消耗您的任何积分。

我们希望您永远不需要考虑积分限制的问题。在极少数情况下如果您达到了限制，您会知道发生了什么，并且我们可以快速为您解决。

<a id='2'></a>
## 2 - Integrating Data into an LLM Prompt

In this section, you will learn how to effectively incorporate data into a prompt before passing it to a Large Language Model (LLM). We will work with a small dataset consisting of JSON files that contain information about houses. It will help you understand how to augment prompts in the context of RAG.
在本节中，您将学习如何有效地将数据整合到提示词中，然后将其传递给大型语言模型（LLM）。我们将使用一个由 JSON 文件组成的小型数据集，这些文件包含有关房屋的信息。这将有助于您理解如何在 RAG（检索增强生成） 的背景下增强提示词。

<a id='2-1'></a>
### 2.1 Understanding the data structure

Let's have a quick look in the data structure. It is a tiny dataset of houses. A list containing one dictionary per house.
让我们快速看一下数据结构。这是一个很小的房屋数据集 (tiny dataset of houses)。它是一个列表 (list)，其中每栋房屋对应一个字典 (dictionary)。

In [13]:
house_data = [
    {
        "address": "123 Maple Street",
        "city": "Springfield",
        "state": "IL",
        "zip": "62701",
        "bedrooms": 3,
        "bathrooms": 2,
        "square_feet": 1500,
        "price": 230000,
        "year_built": 1998
    },
    {
        "address": "456 Elm Avenue",
        "city": "Shelbyville",
        "state": "TN",
        "zip": "37160",
        "bedrooms": 4,
        "bathrooms": 3,
        "square_feet": 2500,
        "price": 320000,
        "year_built": 2005
    }
]

<a id='2-2'></a>
### 2.2 Creating the Prompt

Let's begin by constructing the prompt. The first step is to design a layout for the data.

In [12]:
# First, let's create a layout for the houses

def house_info_layout(houses):
    # Create an empty string
    layout = ''
    # Iterate over the houses
    for house in houses:
        # For each house, append the information to the string using f-strings
        # The following way using brackets is a good way to make the code readable as in each line you can start a new f-string that will appended to the previous one
        layout += (f"House located at {house['address']}, {house['city']}, {house['state']} {house['zip']} with "
            f"{house['bedrooms']} bedrooms, {house['bathrooms']} bathrooms, "
            f"{house['square_feet']} sq ft area, priced at ${house['price']}, "
            f"built in {house['year_built']}.\n") # Don't forget the new line character at the end!
    return layout

In [14]:
# Check the layout
print(house_info_layout(house_data))

House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.



Now create a function that generates the prompt to be passed to the Language Learning Model (LLM). The function will take a user-provided query and the available housing data as inputs to effectively address the user's query.

现在创建一个函数 (function)，用于生成将传递给大型语言模型 (LLM) 的提示词 (prompt)。该函数将把用户提供的查询 (user-provided query) 和可用的房屋数据 (available housing data) 作为输入，以便有效地解决用户的查询。

In [15]:
def generate_prompt(query, houses):
    # The code made above is modular enough to accept any list of houses, so you could also choose a subset of the dataset.
    # This might be useful in a more complex context where you want to give only some information to the LLM and not the entire data
    houses_layout = house_info_layout(houses)
    # Create a hard-coded prompt. You can use three double quotes (") in this cases, so you don't need to worry too much about using single or double quotes and breaking the code
    PROMPT = f"""
Use the following houses information to answer users queries.
{houses_layout}
Query: {query}    
             """
    return PROMPT

In [16]:
print(generate_prompt("What is the most expensive house?", houses = house_data))


Use the following houses information to answer users queries.
House located at 123 Maple Street, Springfield, IL 62701 with 3 bedrooms, 2 bathrooms, 1500 sq ft area, priced at $230000, built in 1998.
House located at 456 Elm Avenue, Shelbyville, TN 37160 with 4 bedrooms, 3 bathrooms, 2500 sq ft area, priced at $320000, built in 2005.

Query: What is the most expensive house?    
             


Now let's call the LLM!

In [ ]:
query = "What is the most expensive house? And the bigger one?"
# Asking without the augmented prompt, let's pass the role as user
query_without_house_info = generate_with_single_input(prompt = query, role = 'user')
# With house info, given the prompt structuer, let's pass the role as assistant
enhanced_query = generate_prompt(query, houses = house_data)
query_with_house_info = generate_with_single_input(prompt = enhanced_query, role = 'assistant')

In [ ]:
# Without house info
print(query_without_house_info['content'])

In [ ]:
# With house info
print(query_with_house_info['content'])

Keep it up! You finished the introductory ungraded lab on how to call LLMs and augment prompts!
继续努力！您已经完成了关于如何调用大型语言模型 (LLMs) 和增强提示词的入门级未评分实验！